# 02 - QLoRA fine-tuning with UnslothBase model: `Qwen/Qwen2.5-Coder-3B-Instruct`. Train on the ChatML dataset built in 01.Hardware notes (recommended): a single CUDA GPU with >= 10 GB VRAM (RTX 3060 12 GB and up), or Colab T4. On Windows use WSL2 with an NVIDIA driver.

In [ ]:
# %%capture
!pip install -U pip unsloth "torch>=2.3" "bitsandbytes>=0.43" "trl>=0.9" "peft>=0.12"

In [ ]:
import torch
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    "Qwen/Qwen2.5-Coder-3B-Instruct",
    max_seq_length=4096,
    dtype=torch.bfloat16,
    load_in_4bit=True,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.0,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
)
print(model.print_trainable_parameters())

In [ ]:
from datasets import load_dataset, Dataset
from trl import SFTConfig, SFTTrainer

def format_example(example):
    return {"text": example["text"]}

train_ds = Dataset.from_json("data/chatml/train.jsonl").map(format_example)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    dataset_text_field="text",
    max_seq_length=4096,
    dataset_num_proc=8,
    args=SFTConfig(
        per_device_train_batch_size=4,
        gradient_accumulation_steps=8,
        learning_rate=2e-4,
        lr_scheduler_type="linear",
        warmup_ratio=0.08,
        num_train_epochs=2,
        logging_steps=10,
        max_steps=-1,
        output_dir="checkpoints/zeroerr-3b-lora",
        save_strategy="epoch",
    ),
)
trainer_stats = trainer.train()

After training, run the export notebook (03) to produce merged 16-bit weights and the Q4_K_M GGUF used by Ollama.